# Gradient Analysis — f0-register stratification (Reviewer 4 #2)

Reviewer 4 asks to *"stratify Table 1 / Table 2 by f0 register — at minimum
{D1–D2, D3–D4, D5–D6}"*, predicting that **onset-time gradients improve toward
the treble**: short-time power spectra resolve onset at the window level (a few
cycles of the fundamental), so high-f0 transients carry usable onset
information while bass events do not.

We test this for the paper's losses (**MSS**, **SOT**) and both synths
(**tKSA**, **fKSA**), STE excitation. Only the **onset-time** parameter is
stratified — the other parameters are essentially f0-insensitive in the
original tables (decay/a1/dynamic/burst_gain ≈ 100%, and f0 is the swept axis).

- **Table 1** (LTI, single event): onset CGA/FGA at the six D-notes, grouped
  into the three registers, plus a finer f0 sweep for the trend curve.
- **Table 2** (LTV, multi-event): onset *Any*/*Joint* accuracy at each register.


In [ ]:
import sys
from pathlib import Path

_root = Path.cwd()
while not (_root / "src").is_dir() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from itertools import product as iterproduct
from scipy.optimize import linear_sum_assignment

from src.synths.synth import Synth, SynthConfig
from src.synths.ddsp import Implementation, ExcitationMode
from src.losses import MultiScaleSpectralLoss, SOT2048Loss

FS = 16000
NUM_SAMPLES = FS * 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_INIT = 200
BATCH_SIZE = 32
FGA_FRACTION = 0.10
TGT_TIME = 0.5
print("Device:", DEVICE)

# Fixed (non-f0) ground-truth parameters; only f0 varies across registers.
GT = {"a1": 0.2, "decay": 0.99, "pluck_position": 0.25,
      "burst_gain": 0.9, "dynamic_level": 0.9}

# Reviewer's registers: the six D-notes grouped two-per-register.
D_NOTES = {"D1": 36.71, "D2": 73.42, "D3": 146.83,
           "D4": 293.66, "D5": 587.33, "D6": 1174.66}
REGISTERS = {"D1-D2 (bass)": ["D1", "D2"],
             "D3-D4 (mid)":  ["D3", "D4"],
             "D5-D6 (treble)": ["D5", "D6"]}

# STE excitation, paper losses.
def _build(ks, *, lti):
    kw = dict(num_samples=NUM_SAMPLES, fs=FS, implementation=ks,
              excitation_mode=ExcitationMode.STE)
    if ks == Implementation.FREQUENCY_SAMPLING:
        if lti:
            kw["use_lti"] = True
        else:
            kw.update(n_fft=16384, hop_length=256, use_lti=False)
    return Synth(SynthConfig(**kw)).to(DEVICE)

SYNTHS_LTI = {"tKSA": _build(Implementation.TIME_DOMAIN, lti=True),
              "fKSA": _build(Implementation.FREQUENCY_SAMPLING, lti=True)}
SYNTHS_LTV = {"tKSA": _build(Implementation.TIME_DOMAIN, lti=False),
              "fKSA": _build(Implementation.FREQUENCY_SAMPLING, lti=False)}
ORACLE = SYNTHS_LTI["tKSA"]

MSS = MultiScaleSpectralLoss().to(DEVICE)
SOT = SOT2048Loss(sample_rate=FS).to(DEVICE)
LOSSES = {"MSS": MSS, "SOT": SOT}

def make_params(f0, times, K=1):
    """times: list[float] | Tensor[B,K]."""
    if isinstance(times, (list, tuple)):
        times = torch.tensor([times], dtype=torch.float32, device=DEVICE)
    B = times.shape[0]
    return {"exists": torch.ones(B, K, device=DEVICE), "time": times,
            "f0": torch.full((B, K), f0, device=DEVICE),
            "burst_gain": torch.full((B, K), GT["burst_gain"], device=DEVICE),
            "pluck_position": torch.full((B, K), GT["pluck_position"], device=DEVICE),
            "dynamic_level": torch.full((B, K), GT["dynamic_level"], device=DEVICE),
            "a1": torch.full((B, K), GT["a1"], device=DEVICE),
            "decay": torch.full((B, K), GT["decay"], device=DEVICE)}


---
## Table 1 — single-event onset accuracy vs register (LTI)

For each f0, sweep the onset time over the full range (CGA) and a ±10% band
(FGA), and record the fraction of inits whose gradient points toward the target
onset (0.5).

In [ ]:
def onset_accuracy(synth, loss_fn, f0, target_audio):
    """CGA (full range) and FGA (+-10% band) of the onset-time gradient."""
    def grads(vals):
        g = np.empty(len(vals))
        for s in range(0, len(vals), BATCH_SIZE):
            e = min(s + BATCH_SIZE, len(vals)); B = e - s
            p = make_params(f0, [TGT_TIME])
            p = {k: v.expand(B, -1).clone().detach() for k, v in p.items()}
            t = torch.tensor(vals[s:e], dtype=torch.float32, device=DEVICE).unsqueeze(1)
            t.requires_grad_(True); p["time"] = t
            y, _ = synth(p)
            L = loss_fn(y, target_audio.expand(B, -1)); L.backward()
            g[s:e] = t.grad[:, 0].detach().cpu().numpy()
        return g
    def acc(vals):
        d = TGT_TIME - vals; m = np.abs(d) > 1e-8
        return float(((grads(vals)[m] * d[m]) < 0).mean())
    cga = acc(np.linspace(0.01, 0.99, N_INIT))
    lo = max(0.01, TGT_TIME - FGA_FRACTION); hi = min(0.99, TGT_TIME + FGA_FRACTION)
    fga = acc(np.linspace(lo, hi, N_INIT))
    return cga, fga

# Per-note accuracy at the six D-notes.
note_rows = {}
for note, f0 in tqdm(D_NOTES.items(), desc="D-notes"):
    with torch.no_grad():
        tgt, _ = ORACLE.oracle_synth(make_params(f0, [TGT_TIME]))
    for lname, lf in LOSSES.items():
        for sname, syn in SYNTHS_LTI.items():
            note_rows[(note, lname, sname)] = onset_accuracy(syn, lf, f0, tgt)

# Group into registers (mean over the two notes).
rows = []
for reg, notes in REGISTERS.items():
    row = {"Register": reg}
    for lname in LOSSES:
        for sname in SYNTHS_LTI:
            cga = np.mean([note_rows[(n, lname, sname)][0] for n in notes])
            fga = np.mean([note_rows[(n, lname, sname)][1] for n in notes])
            row[f"{lname}.{sname}.CGA"] = f"{cga:.0%}"
            row[f"{lname}.{sname}.FGA"] = f"{fga:.0%}"
    rows.append(row)
df_t1 = pd.DataFrame(rows).set_index("Register")
print(df_t1.to_string())
df_t1.to_csv("gradient_onset_f0_stratified_t1.csv")


In [ ]:
# Finer f0 sweep for the trend curve.
f0_sweep = np.geomspace(D_NOTES["D1"], D_NOTES["D6"], 13)
curve = {(l, s): [] for l in LOSSES for s in SYNTHS_LTI}
for f0 in tqdm(f0_sweep, desc="f0 sweep"):
    with torch.no_grad():
        tgt, _ = ORACLE.oracle_synth(make_params(float(f0), [TGT_TIME]))
    for lname, lf in LOSSES.items():
        for sname, syn in SYNTHS_LTI.items():
            curve[(lname, sname)].append(onset_accuracy(syn, lf, float(f0), tgt))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, metric, mi in zip(axes, ["CGA", "FGA"], [0, 1]):
    for (lname, sname), vals in curve.items():
        ax.plot(f0_sweep, [v[mi] for v in vals], marker="o", label=f"{lname}.{sname}")
    for f0 in D_NOTES.values():
        ax.axvline(f0, color="0.85", lw=0.8, zorder=0)
    ax.axhline(0.5, color="k", ls=":", lw=1, label="chance")
    ax.set_xscale("log"); ax.set_xlabel("f0 (Hz)"); ax.set_title(f"Onset {metric} vs f0")
    ax.set_xticks(list(D_NOTES.values())); ax.set_xticklabels(list(D_NOTES), rotation=0)
axes[0].set_ylabel("onset gradient accuracy"); axes[0].legend(fontsize=8)
fig.suptitle("Onset-time gradient accuracy across the register (bass → treble)")
fig.tight_layout(); plt.show()


---
## Table 2 — multi-event onset accuracy vs register (LTV)

*Any* = each event's gradient points toward *some* target; *Joint* = all K
simultaneously correct. Evaluated at each register's geometric-center pitch.

In [ ]:
def event_times(K):
    return [k / K - 1 / (2 * K) for k in range(1, K + 1)]

def make_init_grid(K, n_target=N_INIT, lo=0.01, hi=0.99):
    n = max(2, int(round(n_target ** (1.0 / K))))
    return np.array(list(iterproduct(*([np.linspace(lo, hi, n)] * K))))

def hungarian_match(pred, tgt):
    r, c = linear_sum_assignment(np.abs(pred[:, None] - tgt[None, :]))
    a = np.empty(len(pred), dtype=int); a[r] = c; return a

def check_any(g, pred, tgt, eps=1e-12):
    ok = np.zeros(len(g), bool)
    for i in range(len(g)):
        if abs(g[i]) < eps: continue
        ok[i] = np.any(tgt > pred[i] + eps) if -g[i] > 0 else np.any(tgt < pred[i] - eps)
    return ok

def check_matched(g, pred, tgt, asg, eps=1e-8):
    ok = np.zeros(len(g), bool)
    for i in range(len(g)):
        d = tgt[asg[i]] - pred[i]
        if abs(d) < eps: ok[i] = True
        elif abs(g[i]) >= 1e-12: ok[i] = (g[i] * d) < 0
    return ok

EVENT_COUNTS = [2, 3, 4]
REG_CENTER = {reg: float(np.sqrt(D_NOTES[notes[0]] * D_NOTES[notes[1]]))
              for reg, notes in REGISTERS.items()}

rows = []
for reg, f0 in tqdm(REG_CENTER.items(), desc="Table 2 registers"):
    for K in EVENT_COUNTS:
        tgt_t = np.array(event_times(K))
        with torch.no_grad():
            tgt_audio, _ = ORACLE.oracle_synth(make_params(f0, event_times(K), K=K))
        grid = make_init_grid(K); nc = grid.shape[0]
        row = {"Register": reg, "f0": round(f0, 1), "K": K}
        for lname, lf in LOSSES.items():
            for sname, syn in SYNTHS_LTV.items():
                anys, joints = [], []
                for s in range(0, nc, BATCH_SIZE):
                    e = min(s + BATCH_SIZE, nc); B = e - s
                    pr = grid[s:e]
                    t = torch.tensor(pr, dtype=torch.float32, device=DEVICE, requires_grad=True)
                    p = make_params(f0, t, K=K)
                    y, _ = syn(p); L = lf(y, tgt_audio.expand(B, -1)); L.backward()
                    gnp = t.grad.detach().cpu().numpy()
                    for b in range(B):
                        a = hungarian_match(pr[b], tgt_t)
                        anys.append(check_any(gnp[b], pr[b], tgt_t).mean())
                        joints.append(check_matched(gnp[b], pr[b], tgt_t, a).all())
                row[f"{lname}.{sname}.Any"] = f"{np.mean(anys):.0%}"
                row[f"{lname}.{sname}.Joint"] = f"{np.mean(joints):.0%}"
        rows.append(row)
df_t2 = pd.DataFrame(rows).set_index(["Register", "K"])
print(df_t2.to_string())
df_t2.to_csv("gradient_onset_f0_stratified_t2.csv")


---
## Takeaways

Read against Reviewer 4's prediction (onset gradients should improve toward the
treble). If the curve and tables rise from bass → treble, that *confirms* the
prediction and motivates a multi-resolution / onset-aware loss for the bass
register; if onset accuracy stays at chance across registers, the insensitivity
is uniform and the bass-vs-treble framing should be softened. (Fill in once
run.)